# BloodMNIST predictions
## Run 1: ResNet18 at 224 × 224 (nearest-neighbor resize), ResNet50 at 28 × 28, original ViT processor

In [1]:
from pathlib import Path
import numpy as np
import torch
from transformers import ViTForImageClassification, ViTImageProcessor
from torchvision.models import resnet18
from models.resnet18.models import ResNet50

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
batch_size = 32
output = Path('predictions')
output.mkdir(exist_ok=True)
print(device)
model_names = ['resnet18', 'resnet50', 'vit']

cuda


## Dataset

In [2]:
with np.load('data/bloodmnist.npz', allow_pickle=False) as data:
    splits = {split: (data[f'{split}_images'], data[f'{split}_labels'].ravel())
              for split in ('train', 'val', 'test')}

## Models

In [3]:
models = [resnet18(weights=None, num_classes=8), ResNet50(in_channels=3, num_classes=8)]
checkpoints = ['weights_bloodmnist/resnet18_224_1.pth', 'weights_bloodmnist/resnet50_28_1.pth', 'models/vit']
for model, path in zip(models, checkpoints):
    model.load_state_dict(torch.load(path, map_location='cpu', weights_only=True)['net'])
models.append(ViTForImageClassification.from_pretrained('models/vit', local_files_only=True))
models = [model.to(device).eval() for model in models]
processor = ViTImageProcessor.from_pretrained('models/vit', local_files_only=True)


## Predictions → CSV (image_index, label, prediction, p0–p7)

In [4]:
@torch.inference_mode()
def predict(m, images):
    probabilities, scores = [], []
    for start in range(0, len(images), batch_size):
        batch = images[start:start + batch_size]
        if model_names[m] != 'vit':
            x = torch.from_numpy(batch.copy()).permute(0, 3, 1, 2).to(device).float()
            if model_names[m] == 'resnet18':
                x = torch.nn.functional.interpolate(x, size=(224, 224), mode='nearest')
            logits = models[m]((x / 255 - 0.5) / 0.5)
        else:
            inputs = processor(images=list(batch), return_tensors='pt').to(device)
            logits = models[m](**inputs).logits
        probabilities.append(logits.softmax(-1).cpu().numpy())
        scores.append(logits.cpu().numpy())
    return np.concatenate(probabilities), np.concatenate(scores)

for m, name in enumerate(model_names):
    for split, (images, labels) in splits.items():
        probabilities, logits = predict(m, images)
        predictions = probabilities.argmax(axis=1)
        rows = np.column_stack((np.arange(len(labels)), labels, predictions, probabilities))
        np.savetxt(output / f'{name}_{split}.csv', rows, delimiter=',',
                   header='image_index,label,prediction,' + ','.join(f'p{i}' for i in range(8)),
                   comments='', fmt=['%d'] * 3 + ['%.9g'] * 8)
        np.savez_compressed(output / f'{name}_{split}.npz', probabilities=probabilities, logits=logits,
                            sample_index=np.arange(len(labels)), labels=labels, model=name,
                            checkpoint=checkpoints[m], split=split, input_resolution=224 if name != 'resnet50' else 28)
        print(f'{name} {split}: {len(labels)} predictions, accuracy {(predictions == labels).mean():.2%}')


resnet18 train: 11959 predictions, accuracy 100.00%


resnet18 val: 1712 predictions, accuracy 97.20%


resnet18 test: 3421 predictions, accuracy 96.61%


resnet50 train: 11959 predictions, accuracy 100.00%


resnet50 val: 1712 predictions, accuracy 96.09%


resnet50 test: 3421 predictions, accuracy 95.73%


vit train: 11959 predictions, accuracy 99.48%


vit val: 1712 predictions, accuracy 97.78%


vit test: 3421 predictions, accuracy 96.99%
